# 9. M0 interpretations: populations of real individuals

Every tutorial so far worked at **M1**, the model level: `part rotors :
Rotor[4]` *describes* a quad-copter as having four rotors. **M0** is the
level below -- the four actual rotor individuals one particular quad
*has*, each with its own identity, its own attribute values, and (for
behaviors) its own lifetime. `longeron.m0` builds such populations
directly on the interpreter: an `Interpretation` is a set of
`Individual` runtime instances with stable `qname#index` ids, and every
query -- sequences, roll-ups, sampling -- runs over those actual
individuals instead of the model's descriptions.

**You will learn how to:**

- build a nominal interpretation and address every individual by its
  stable id (`interpret`, `individuals`);
- read features as KerML Annex A sequences (`sequences`);
- roll expressions up over the actual population, and catch M1
  shortcuts that hand-encode the population size (`rollup`,
  `Interpretation.gaps`);
- sample the space of legal populations with the seeded random
  strategy (`strategy="random"`, `sample`);
- turn a recorded state-machine execution into occurrence individuals
  with lifetimes (`from_timeline`) -- the same representation;
- read a trade-study architecture as a partial interpretation whose
  individuals reproduce the trades metrics (`from_architecture`);
- serialize an interpretation (`to_dict`) -- and know why the standard
  API projection never carries it.

**Prerequisites:** tutorial 3 (instantiation and expressions); tutorial
4's state machine and tutorial 7's trade study both return here.
`longeron.m0` itself is stdlib-only -- only the trade-study bridge cell
uses `longeron.analysis`.

In [ ]:
import longeron
from longeron import m0

## A population with names

Here is a deliberately small inline fleet. Note the `totalMass`
expression: it hand-encodes the rotor count (`4.0 * 0.06`), exactly as
`examples/drone.sysml` does -- keep an eye on it. `m0.interpret` builds
the **nominal** population: exact multiplicities expand fully, ranges
take their lower bound (the same choice `Interpreter.instantiate`
makes), and every individual gets a stable `qname#index` id -- the root
is `#0`, nested populations index per feature, singletons omit the
index.

What to look for in the output: four *distinct* rotor individuals, each
addressable by id; the unvalued `livery` attribute stays `None` under
nominal (the random strategy fills it later); and `gaps` is empty --
this population evaluated completely.

In [ ]:
FLEET = """
package Fleet {
    enum def Livery { plain; racing; stealth; }
    part def Rotor {
        attribute mass : Real = 0.06;
        attribute livery : Livery;
    }
    part def Battery {
        attribute mass : Real = 0.38;
    }
    part def Quad {
        attribute frameMass : Real = 0.42;
        part battery : Battery;
        part rotors : Rotor[4];
        // the M1 shortcut: the population size is hand-encoded
        attribute totalMass : Real = frameMass + battery.mass + 4.0 * 0.06;
    }
    part def FieldQuad {
        part rotors : Rotor[2..6];
        part spares : Rotor[0..*];
    }
}
"""

fleet = longeron.loads(FLEET)
quad = m0.interpret(fleet, "Fleet::Quad")
print("root:   ", quad.root)
print("battery:", quad.root.slots["battery"].id, " (a singleton omits the index)")
for rotor in quad.root.slots["rotors"]:
    print(f"  {rotor.id}  mass={rotor.slots['mass']}  livery={rotor.slots['livery']}")
rotor_count = len(quad.individuals("Fleet::Rotor"))
print("individuals:", len(quad.individuals()), " of which rotors:", rotor_count)
print("gaps:", quad.gaps)

### A feature reads as a set of sequences (KerML Annex A)

Annex A of the KerML specification interprets a feature as a set of
*sequences* whose prefix is an individual of the featuring type:
`sequences("rotors")` yields `(quad, rotor_i)` pairs, and the nested
`sequences("rotors.mass")` extends each pair by one more step.

What to look for: four 3-tuples, one per actual rotor, each ending in
that rotor's own value.

In [ ]:
for seq in quad.sequences("rotors")[:2]:
    print(seq)
print("...\n")
for owner, rotor, mass in quad.sequences("rotors.mass"):
    print(f"({owner.id}, {rotor.id}, {mass})")

## Roll-ups aggregate the individuals that actually exist

`rollup` evaluates an expression with feature references resolved
against the root individual's slots, so `sum(rotors.mass)` adds the
four *real* rotors. The M1 expression cannot do that: `totalMass`
hand-encodes the count (`4.0 * 0.06`) because at M1 there is no
population to sum over.

What to look for: at multiplicity `[4]` the two routes agree -- but
only because the hand-encoded `4.0` happens to match the population.

In [ ]:
print("sum over the actual rotors:", quad.rollup("sum(rotors.mass)"))
print("the declared M1 shortcut:  ", quad.rollup("totalMass"))
print("M0 total mass:             ", quad.rollup("frameMass + battery.mass + sum(rotors.mass)"))

### Change the multiplicity and the shortcut lies; `gaps` tells the truth

Stretch the same model to `rotors : Rotor[6]` and re-interpret. The M1
shortcut still evaluates -- to the same, now wrong, number, silently.
The M0 roll-up weighs what is actually there.

There is a second way to write the shortcut: the *per-unit convention*
(`4.0 * rotors.mass`) that `examples/drone_catalog.sysml` uses for its
trade study. Over a real population `rotors.mass` is six values, and
multiplying `4.0` by a list is not an answer. The ratified design
decision: M1/M0 divergence **degrades to `None` and lands in
`Interpretation.gaps` -- it never raises**. The population still
builds, the hole is honest, and a strict caller simply asserts
`gaps == []`.

In [ ]:
hexa = m0.interpret(longeron.loads(FLEET.replace("Rotor[4]", "Rotor[6]")), "Fleet::Quad")
print("rotors:", len(hexa.root.slots["rotors"]))
print("the M1 shortcut still claims:", hexa.root.slots["totalMass"])
m0_total = hexa.rollup("frameMass + battery.mass + sum(rotors.mass)")
print("M0 total over 6 rotors:      ", round(m0_total, 3))

# the same model in the per-unit convention drone_catalog.sysml uses
convention = FLEET.replace("Rotor[4]", "Rotor[6]").replace("4.0 * 0.06", "4.0 * rotors.mass")
honest = m0.interpret(longeron.loads(convention), "Fleet::Quad")
print("\nper-unit convention over a real population:")
print("totalMass slot:", honest.root.slots["totalMass"])
print("gaps:", honest.gaps)

## Nominal takes the lower bound; random explores the range

`FieldQuad` declares `rotors : Rotor[2..6]` and `spares : Rotor[0..*]`.
Under nominal both take their lower bound -- conservative,
deterministic, and consistent with `instantiate()`; that default is the
other ratified decision. `strategy="random"` draws population sizes
uniformly within the bounds (an unbounded upper is capped at
`lower + 3`), samples unvalued enum and Boolean attributes from their
literal domains, and is fully seeded: equal seeds reproduce equal
populations, and `sample(n)` derives `n` fresh interpretations from the
parent seed.

What to look for: nominal always answers 2 rotors and 0 spares; the
random draws vary within `[2..6]` and `[0..3]`; and the rotors'
liveries now carry sampled enum values.

In [ ]:
def shape(it):
    return f"{len(it.root.slots['rotors'])} rotors, {len(it.root.slots['spares'])} spares"


nominal = m0.interpret(fleet, "Fleet::FieldQuad")
print("nominal:", shape(nominal))

drawn = m0.interpret(fleet, "Fleet::FieldQuad", strategy="random", seed=7)
print("seed 7: ", shape(drawn))
print("liveries:", [rotor.slots["livery"].name for rotor in drawn.root.slots["rotors"]])
for s in drawn.sample(3):
    print(f"  sample seed {s.seed}: {shape(s)}")

rerun = m0.interpret(fleet, "Fleet::FieldQuad", strategy="random", seed=7)
print("equal seeds reproduce equal populations:", rerun.to_dict() == drawn.to_dict())

## Traces are interpretations

pymbe, the reference implementation for these population semantics,
could describe individuals but never execute anything. longeron already
executes state machines, and a recorded execution *is* an
interpretation of the behavior. `from_timeline` turns every contiguous
state activation of tutorial 4's `Drone::FlightStates` into an
**occurrence individual** (`qname@k`) with `start`/`end`/`duration`
slots, owned by a root that spans the whole recording.

What to look for: the flight reads as a population of five occurrences
in activation order, and the second visit to `idle` gets a fresh
identity, `@1`.

In [ ]:
from longeron.replay import record_timeline

drone = longeron.load("../examples/drone.sysml")
interp = longeron.Interpreter(drone)
flight = record_timeline(
    interp,
    "Drone::FlightStates",
    [1.5, "launch", 2.0, "airborne", 10.0, "low_battery", 1.0, "touchdown"],
)
trace = m0.from_timeline(flight, source="Drone::FlightStates")
print("strategy:", trace.strategy, " recording span:", trace.root.slots["duration"], "s\n")
for occ in trace.root.slots["occurrences"]:
    start, end = occ.slots["start"], occ.slots["end"]
    print(f"  {occ.id:36s} {start:5.1f} -> {end:5.1f}  ({occ.slots['duration']:4.1f} s)")
print("\nidle re-entries:", [ind.id for ind in trace.individuals("Drone::FlightStates::idle")])
print("sum(occurrences.duration):", trace.rollup("sum(occurrences.duration)"))

### One representation, two kinds of individual

A statically populated rotor and a recorded occurrence are the same
class, queried by the same machinery -- only their slots differ, a mass
versus a lifetime. That is why `rollup` and `sequences` work unchanged
over executions: `sum(occurrences.duration)` above is the same
operation as `sum(rotors.mass)`.

In [ ]:
static = m0.interpret(drone, "Drone::QuadCopter").root.slots["rotors"][2]
occurrence = trace.individuals("Drone::FlightStates::flying")[0]
for individual in (static, occurrence):
    print(f"{individual!r}\n    slots: {dict(individual.slots)}")
print("same class:", type(static) is type(occurrence) is m0.Individual)

## A trades `Architecture` is a partial interpretation

Tutorial 7 enumerated architectures by pinning every variation point
and scoring the result through the interpreter. That pinning *is* a
partial M0 interpretation: variant selection fixed, population nominal.
`from_architecture` makes it literal on the quad-copter catalog
(`examples/drone_catalog.sysml`, 54 mixes).

What to look for: the four `motors` become four individuals of the
*selected* variant, and the catalog's M1 metrics -- all written in the
per-unit `4.0 * x` convention -- surface as four honest gaps, exactly
as in the inline model above.

In [ ]:
from longeron.analysis.trades import TradeStudy

catalog = longeron.load("../examples/drone_catalog.sysml")
study = TradeStudy(catalog, "DroneCatalog::TradeQuad")
architectures = study.all_architectures()
feasible = [a for a in architectures if a.verified]
print(len(architectures), "mixes,", len(feasible), "feasible")

endurance = max(feasible, key=lambda a: a.metrics["hoverMinutes"])
print("longest-hover mix:", endurance.selection, "\n")

quad_m0 = m0.from_architecture(study, endurance)
for motor in quad_m0.root.slots["motors"]:
    print(f"  {motor.id}: {motor.type_name}")
print("\nthe M1 metrics use the homogeneous 4.0 * x convention -- honest gaps:")
for gap in quad_m0.gaps:
    print(" ", gap)

### The regression: individuals reproduce the trades metrics

Rewrite each metric over the actual population and it must equal the
number the trades machinery computed at M1 -- same model, two routes,
one answer.

What to look for: four exact matches. The test suite
(`tests/test_m0.py`) asserts this equality for **all 54 mixes**, so the
two semantics cannot drift apart unnoticed.

In [ ]:
import math

ROLLUPS = {
    "totalMass": "frameMass + payloadMass + battery.mass + esc.mass"
    " + sum(motors.mass) + sum(props.mass)",
    "totalCost": "battery.cost + esc.cost + sum(motors.cost) + sum(props.cost)",
    "totalThrust": "sum(motors.maxThrust)",
    "hoverMinutes": "battery.capacity / sum(motors.hoverCurrent) * 60.0",
}
print(f"{'metric':14s}{'trades (M1)':>14s}{'roll-up (M0)':>14s}")
for metric, expr in ROLLUPS.items():
    value = quad_m0.rollup(expr)
    assert math.isclose(value, endurance.metrics[metric], rel_tol=1e-12)
    print(f"{metric:14s}{endurance.metrics[metric]:14.3f}{value:14.3f}")

## The JSON shape stays out of the standard API

`to_dict()` projects an interpretation -- ids, selection, gaps, the
full slot tree -- into plain JSON-able data. It is a deliberate
longeron *extension*: the OMG Systems Modeling API has no M0
representation, so `to_api_json` (tutorial 2) never emits it, keeping
the standard record stream clean for ecosystem consumers -- the third
ratified decision; if interpretations are ever served over HTTP they
enter through an extension namespace instead.

In [ ]:
import json

payload = quad.to_dict()
print("keys:", list(payload))
print(json.dumps(payload["root"]["rotors"][0], indent=2))
print("JSON round-trip:", json.loads(json.dumps(payload)) == payload)

The M0 story in one line: **one `Individual` representation, from
static part populations through seeded random draws to recorded
occurrences and trade-study architectures**, with `rollup` and
`sequences` as the single query surface and `gaps` as the honesty
channel. The design rationale, the three ratified decisions this
tutorial demonstrated, and what comes next (an RDF projection of
individuals, heterogeneous per-index selections feeding back into the
trade study) live in `docs/design/m0-interpretations.md`.